Imports

In [ ]:
!pip -q install pyomo highspy

import pandas as pd
import numpy as np
from pathlib import Path

from pyomo.environ import (
    ConcreteModel, Var, NonNegativeReals, Objective, Constraint, minimize, Set, value
)
from pyomo.contrib.appsi.solvers import Highs


Generate Mock Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


# 1) CREATE MOCK PATTERN MASTER FILE
INPUT_DIR = Path("inputs")
INPUT_DIR.mkdir(exist_ok=True)
patterns = [
    # Dominant single SKU patterns
    {"Cut Pattern":"P1","Sku A":"1001","Yield A":0.86,"Sku B":"","Yield B":0.00,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":3000},
    {"Cut Pattern":"P2","Sku A":"1023","Yield A":0.85,"Sku B":"","Yield B":0.00,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2950},
    {"Cut Pattern":"P3","Sku A":"2021","Yield A":0.84,"Sku B":"","Yield B":0.00,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2900},
    {"Cut Pattern":"P4","Sku A":"3033","Yield A":0.83,"Sku B":"","Yield B":0.00,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2850},

    # 2 SKU mix
    {"Cut Pattern":"P5","Sku A":"1001","Yield A":0.55,"Sku B":"1023","Yield B":0.28,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2600},
    {"Cut Pattern":"P6","Sku A":"1023","Yield A":0.50,"Sku B":"3033","Yield B":0.30,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2550},
    {"Cut Pattern":"P7","Sku A":"2021","Yield A":0.48,"Sku B":"4043","Yield B":0.32,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2500},
    {"Cut Pattern":"P8","Sku A":"3033","Yield A":0.45,"Sku B":"7072","Yield B":0.34,"Sku C":"","Yield C":0.00,"Belt Speed lbs/hr":2450},

    # 3 SKU patterns
    {"Cut Pattern":"P9","Sku A":"1001","Yield A":0.40,"Sku B":"2021","Yield B":0.24,"Sku C":"7072","Yield C":0.22,"Belt Speed lbs/hr":2300},
    {"Cut Pattern":"P10","Sku A":"1023","Yield A":0.38,"Sku B":"4043","Yield B":0.26,"Sku C":"6066","Yield C":0.20,"Belt Speed lbs/hr":2250},
    {"Cut Pattern":"P11","Sku A":"3033","Yield A":0.36,"Sku B":"5055","Yield B":0.28,"Sku C":"7072","Yield C":0.18,"Belt Speed lbs/hr":2200},
    {"Cut Pattern":"P12","Sku A":"2021","Yield A":0.34,"Sku B":"6066","Yield B":0.30,"Sku C":"7072","Yield C":0.18,"Belt Speed lbs/hr":2150},
]

pattern_df = pd.DataFrame(patterns)

# Calculate trim automatically
pattern_df["Trim"] = (
    1
    - pattern_df["Yield A"]
    - pattern_df["Yield B"]
    - pattern_df["Yield C"]
).round(3).clip(lower=0)

pattern_file = INPUT_DIR / "pattern_master_controlled.csv"
pattern_df.to_csv(pattern_file, index=False)

print("Pattern file written to:", pattern_file.resolve())
display(pattern_df)


# 2) LOAD PATTERN FILE INTO MODEL STRUCTURE
def load_pattern_master_wide_csv(path):

    df = pd.read_csv(path)

    sku_cols = [c for c in df.columns if c.startswith("Sku")]
    yield_cols = [c for c in df.columns if c.startswith("Yield")]

    K = df["Cut Pattern"].astype(str).tolist()

    long_rows = []

    for _, row in df.iterrows():

        k = str(row["Cut Pattern"])
        trim = float(row["Trim"])
        rate_val = float(row["Belt Speed lbs/hr"])

        for sku_col, y_col in zip(sku_cols, yield_cols):

            sku = row[sku_col]
            y = row[y_col]

            if pd.isna(sku) or sku == "":
                continue

            if float(y) <= 0:
                continue

            p = str(int(float(sku)))

            long_rows.append({
                "k": k,
                "p": p,
                "yield": float(y),
                "trim": trim,
                "rate": rate_val
            })

    df_long = pd.DataFrame(long_rows)

    P = sorted(df_long["p"].unique())

    # FIXED ROOT CAUSE:
    # Build dense yield matrix in the order the model expects: Y[(p,k)]
    Y = {(p, k): 0.0 for p in P for k in K}
    for _, row in df_long.iterrows():
        Y[(row["p"], row["k"])] = float(row["yield"])

    waste = {k: float(df.loc[df["Cut Pattern"] == k, "Trim"].iloc[0]) for k in K}
    rate = {k: float(df.loc[df["Cut Pattern"] == k, "Belt Speed lbs/hr"].iloc[0]) for k in K}

    return P, K, Y, waste, rate, df_long

P, K, Y, waste, rate, df_long = load_pattern_master_wide_csv(pattern_file)

print("Loaded:", len(P), "SKUs,", len(K), "patterns")
display(df_long.head())


# 3) TIME HORIZON
T = list(range(1, 29))


# 4) WIP + Line Capacity
wip = {t: 750000.0 for t in T}
use_capacity = True

L = ["Line1", "Line2", "Line3"]
hours = {(l, t): 16.0 for l in L for t in T}


# Controlled WEEKLY demand scenarios
shifts_per_week = 7
T = list(range(1, 29))  # 28 shifts
week_of_t = {t: (t - 1) // shifts_per_week + 1 for t in T}
W = sorted(set(week_of_t.values()))

SCENARIO = "ramp"

base_weekly = {
    "1001": 12000,
    "1023": 10000,
    "2021": 10000,
    "3033": 9500,
    "4043": 5000,
    "5055": 5000,
    "6066": 5000,
    "7072": 5000,
}

weekly_demand = {}
for p in P:
    p = str(p)
    for w in W:
        val = base_weekly.get(p, 0.0)

        if SCENARIO == "baseline":
            mult = 1.0
        elif SCENARIO == "ramp":
            mult = 1.0 + 0.06 * (w - 1)
        elif SCENARIO == "spike":
            mult = 1.35 if w == 3 else 1.0
        elif SCENARIO == "mix_shift":
            if p in ["1001", "1023"]:
                mult = 1.20
            elif p in ["5055", "6066"]:
                mult = 0.85
            else:
                mult = 1.0
        else:
            raise ValueError("Unknown SCENARIO")

        weekly_demand[(p, int(w))] = float(val * mult)

print("Scenario:", SCENARIO)
print("Weeks:", W)
print("Example weekly demand for 1001:",
      {w: weekly_demand[("1001", w)] for w in W})


# OPTIONAL: demand scaling
approx_total_input = sum(wip[t] for t in T)

all_yields = [float(v) for v in Y.values() if float(v) > 0]
avg_yield = np.mean(all_yields) if len(all_yields) > 0 else 0.5

rough_output_cap = approx_total_input * avg_yield * 0.55
total_weekly_demand = sum(weekly_demand.values())

if total_weekly_demand > rough_output_cap and total_weekly_demand > 0:
    scale = rough_output_cap / total_weekly_demand
    print(f"Scaling weekly demand by {scale:.3f} for feasibility.")
    for key in weekly_demand:
        weekly_demand[key] *= scale

# SKU values
sku_value = {
    "1001": 3.20,
    "1023": 3.05,
    "2021": 2.95,
    "3033": 2.85,
    "4043": 2.60,
    "5055": 2.40,
    "6066": 2.35,
    "7072": 2.55,
}

value_per_lb = {}
for k in K:
    k = str(k)
    value_per_lb[k] = sum(
        float(Y.get((str(p), k), 0.0)) * float(sku_value.get(str(p), 0.0))
        for p in P
    )

print("Example value_per_lb:")
for k in list(K)[:5]:
    print(k, round(value_per_lb[str(k)], 4))


# Shift-level demand
demand = {}
for p in P:
    p = str(p)
    for t in T:
        w = week_of_t[t]
        demand[(p, t)] = weekly_demand[(p, w)] / shifts_per_week


# Initial inventory
initial_inventory = {
    str(p): weekly_demand[(str(p), 1)] * 0.60 for p in P
}


# Max delay by SKU
max_delay = {
    str(p): 7 for p in P
}
L_delay = max_delay.copy()


# Preferred line matrix
A = {}

for i, k in enumerate(K):
    preferred_line = L[i % len(L)]
    for l in L:
        A[(str(k), str(l))] = 1 if l == preferred_line else 0

print("Example preferred lines:")
for k in list(K)[:6]:
    print(k, {l: A[(str(k), str(l))] for l in L})


# Penalties
gamma = 0.01
beta = 1.0

Pattern file written to: /content/inputs/pattern_master_controlled.csv


,Cut Pattern,Sku A,Yield A,Sku B,Yield B,Sku C,Yield C,Belt Speed lbs/hr,Trim
0,P1,1001,0.86,,0.00,,0.00,3000,0.14
1,P2,1023,0.85,,0.00,,0.00,2950,0.15
2,P3,2021,0.84,,0.00,,0.00,2900,0.16
3,P4,3033,0.83,,0.00,,0.00,2850,0.17
4,P5,1001,0.55,1023,0.28,,0.00,2600,0.17
5,P6,1023,0.50,3033,0.30,,0.00,2550,0.20
6,P7,2021,0.48,4043,0.32,,0.00,2500,0.20
7,P8,3033,0.45,7072,0.34,,0.00,2450,0.21
8,P9,1001,0.40,2021,0.24,7072,0.22,2300,0.14
9,P10,1023,0.38,4043,0.26,6066,0.20,2250,0.16


Loaded: 8 SKUs, 12 patterns


,k,p,yield,trim,rate
0,P1,1001,0.86,0.14,3000.0
1,P2,1023,0.85,0.15,2950.0
2,P3,2021,0.84,0.16,2900.0
3,P4,3033,0.83,0.17,2850.0
4,P5,1001,0.55,0.17,2600.0


Scenario: ramp
Weeks: [1, 2, 3, 4]
Example weekly demand for 1001: {1: 12000.0, 2: 12720.0, 3: 13440.000000000002, 4: 14160.0}
Example value_per_lb:
P1 2.752
P2 2.5925
P3 2.478
P4 2.3655
P5 2.614
Example preferred lines:
P1 {'Line1': 1, 'Line2': 0, 'Line3': 0}
P2 {'Line1': 0, 'Line2': 1, 'Line3': 0}
P3 {'Line1': 0, 'Line2': 0, 'Line3': 1}
P4 {'Line1': 1, 'Line2': 0, 'Line3': 0}
P5 {'Line1': 0, 'Line2': 1, 'Line3': 0}
P6 {'Line1': 0, 'Line2': 0, 'Line3': 1}


Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1) LOAD PATTERN MASTER FILE
INPUT_DIR = Path("inputs")
pattern_file = INPUT_DIR / "pattern_master_controlled.csv"

def load_pattern_master_wide_csv(path):

    df = pd.read_csv(path)

    sku_cols = [c for c in df.columns if c.startswith("Sku")]
    yield_cols = [c for c in df.columns if c.startswith("Yield")]

    K = df["Cut Pattern"].astype(str).tolist()

    long_rows = []

    for _, row in df.iterrows():

        k = str(row["Cut Pattern"])
        trim = float(row["Trim"])
        rate = float(row["Belt Speed lbs/hr"])

        for sku_col, y_col in zip(sku_cols, yield_cols):

            sku = row[sku_col]
            y = row[y_col]

            if pd.isna(sku) or sku == "":
                continue

            if float(y) <= 0:
                continue

            p = str(int(float(sku)))

            long_rows.append({
                "k": k,
                "p": p,
                "yield": float(y),
                "trim": trim,
                "rate": rate
            })

    df_long = pd.DataFrame(long_rows)

    P = sorted(df_long["p"].unique())

    yld = {(row["k"], row["p"]): row["yield"] for _, row in df_long.iterrows()}

    waste = {k: float(df.loc[df["Cut Pattern"] == k, "Trim"].iloc[0]) for k in K}

    rate = {k: float(df.loc[df["Cut Pattern"] == k, "Belt Speed lbs/hr"].iloc[0]) for k in K}

    return P, K, yld, waste, rate, df_long, df

P, K, yld, waste, rate, df_long, pattern_df = load_pattern_master_wide_csv(pattern_file)

print("Loaded:", len(P), "SKUs,", len(K), "patterns")
print(df_long.head())


# 2) TIME HORIZON
T = list(range(1, 29))


# 3) WIP + Line Capacity
wip = {t: 750000.0 for t in T}

L = ["Line1", "Line2", "Line3"]
hours = {(l, t): 16.0 for l in L for t in T}

# 4) LOAD / CREATE DEMAND
shifts_per_week = 7
week_of_t = {t: (t - 1) // shifts_per_week + 1 for t in T}
W = sorted(set(week_of_t.values()))

# if you have a weekly demand csv, replace this block with a real file load
base_weekly = {
    "1001": 12000,
    "1023": 10000,
    "2021": 10000,
    "3033": 9500,
    "4043": 5000,
    "5055": 5000,
    "6066": 5000,
    "7072": 5000,
}

weekly_demand = {}
for p in P:
    p = str(p)
    for w in W:
        weekly_demand[(p, int(w))] = float(base_weekly.get(p, 0.0))

demand = {}
for p in P:
    p = str(p)
    for t in T:
        w = week_of_t[t]
        demand[(p, t)] = weekly_demand[(p, w)] / shifts_per_week

# 5) VALUES
sku_value = {
    "1001": 3.20,
    "1023": 3.05,
    "2021": 2.95,
    "3033": 2.85,
    "4043": 2.60,
    "5055": 2.40,
    "6066": 2.35,
    "7072": 2.55,
}

value_per_lb = {}
for k in K:
    k = str(k)
    value_per_lb[k] = sum(
        float(yld.get((k, str(p)), 0.0)) * float(sku_value.get(str(p), 0.0))
        for p in P
    )


# 6) INVENTORY / DELAY
initial_inventory = {
    str(p): weekly_demand[(str(p), 1)] * 0.60 for p in P
}

max_delay = {
    str(p): 7 for p in P
}

L_delay = max_delay.copy()


# 7) PREFERRED LINE MATRIX
A = {}

for i, k in enumerate(K):
    preferred_line = L[i % len(L)]
    for l in L:
        A[(str(k), str(l))] = 1 if l == preferred_line else 0


# 8) PENALTIES
gamma = 0.01
beta = 1.0

Loaded: 8 SKUs, 12 patterns
    k     p  yield  trim    rate
0  P1  1001   0.86  0.14  3000.0
1  P2  1023   0.85  0.15  2950.0
2  P3  2021   0.84  0.16  2900.0
3  P4  3033   0.83  0.17  2850.0
4  P5  1001   0.55  0.17  2600.0


Build Model

In [ ]:
from pyomo.environ import *

def build_model(P_set, T_set, K_set, L_set,
                WIP, D, Y, V, R, H, A,
                L_delay,
                gamma=1.0, beta=1.0):
    """
    P_set   : iterable of SKUs p
    T_set   : iterable of days t
    K_set   : iterable of decisions k
    L_set   : iterable of lines l

    WIP[t]      : lbs breast available on day t
    D[p,t]      : demand for sku p on day t
    Y[p,k]      : lbs of sku p produced by decision k per lb assigned
    V[k]        : value per lb of decision k
    R[k]        : lbs/hour for decision k
    H[l,t]      : hours available on line l, day t
    A[k,l]      : 1 if line l is preferred for decision k, else 0
    L_delay[p]  : max allowed delay for sku p

    gamma       : penalty on |P_pt - D_pt|
    beta        : penalty on allowing non-preferred line use
    """

    m = ConcreteModel()

    # Sets
    m.P = Set(initialize=list(P_set), ordered=True)
    m.T = Set(initialize=list(T_set), ordered=True)
    m.K = Set(initialize=list(K_set), ordered=True)
    m.L = Set(initialize=list(L_set), ordered=True)


    # Parameters
    m.WIP = Param(m.T, initialize=WIP, within=NonNegativeReals)
    m.D   = Param(m.P, m.T, initialize=D, within=NonNegativeReals)
    m.Y   = Param(m.P, m.K, initialize=Y, within=NonNegativeReals)
    m.V   = Param(m.K, initialize=V, within=Reals)
    m.R   = Param(m.K, initialize=R, within=PositiveReals)
    m.H   = Param(m.L, m.T, initialize=H, within=NonNegativeReals)
    m.A   = Param(m.K, m.L, initialize=A, within=Binary)
    m.Lag = Param(m.P, initialize=L_delay, within=NonNegativeIntegers)


    # Decision Variables
    # X_klt = lbs assigned to decision k on line l on day t
    m.x = Var(m.K, m.L, m.T, domain=NonNegativeReals)

    # P_pt = lbs of sku p produced on day t
    m.prod = Var(m.P, m.T, domain=NonNegativeReals)

    # z_kt = 1 if decision k is allowed to use a non-preferred line on day t
    m.z = Var(m.K, m.T, domain=Binary)

    # dev_pt = |P_pt - D_pt| linearization variable
    m.dev = Var(m.P, m.T, domain=NonNegativeReals)


    # Constraints
    # 1) WIP availability by day
    def wip_rule(m, t):
        return sum(m.x[k, l, t] for k in m.K for l in m.L) <= m.WIP[t]
    m.WIPConstraint = Constraint(m.T, rule=wip_rule)

    # 2) Line-hour capacity by line and day
    def line_capacity_rule(m, l, t):
        return sum(m.x[k, l, t] / m.R[k] for k in m.K) <= m.H[l, t]
    m.LineCapacityConstraint = Constraint(m.L, m.T, rule=line_capacity_rule)

    # 3) Preferred-line logic
    # If A[k,l] = 1, line is preferred and always allowed
    # If A[k,l] = 0, line is non-preferred and only allowed if z[k,t] = 1
    def line_preference_rule(m, k, l, t):
        return m.x[k, l, t] <= m.R[k] * m.H[l, t] * (m.A[k, l] + m.z[k, t])
    m.LinePreferenceConstraint = Constraint(m.K, m.L, m.T, rule=line_preference_rule)

    # 4) Production definition
    def production_rule(m, p, t):
        return m.prod[p, t] == sum(m.Y[p, k] * m.x[k, l, t] for k in m.K for l in m.L)
    m.ProductionConstraint = Constraint(m.P, m.T, rule=production_rule)

    # 5) Delayed demand coverage
    # sum_{r=1}^t P_pr >= sum_{r=1}^{t-L_p} D_pr, for t > L_p
    T_list = list(m.T.data())

    def delayed_demand_rule(m, p, t):
        lag = int(value(m.Lag[p]))
        t_idx = T_list.index(t)

        if t_idx < lag:
            return Constraint.Skip

        lhs_days = T_list[:t_idx + 1]
        rhs_days = T_list[:t_idx + 1 - lag]

        return sum(m.prod[p, r] for r in lhs_days) >= sum(m.D[p, r] for r in rhs_days)

    m.DelayedDemandConstraint = Constraint(m.P, m.T, rule=delayed_demand_rule)

    # 6) Absolute deviation linearization for |P_pt - D_pt|
    def dev_pos_rule(m, p, t):
        return m.dev[p, t] >= m.prod[p, t] - m.D[p, t]
    m.DevPosConstraint = Constraint(m.P, m.T, rule=dev_pos_rule)

    def dev_neg_rule(m, p, t):
        return m.dev[p, t] >= m.D[p, t] - m.prod[p, t]
    m.DevNegConstraint = Constraint(m.P, m.T, rule=dev_neg_rule)


    # Objective Function
    def objective_rule(m):
        production_value = sum(m.V[k] * m.x[k, l, t] for k in m.K for l in m.L for t in m.T)
        demand_variability_penalty = gamma * sum(m.dev[p, t] for p in m.P for t in m.T)
        nonpreferred_line_penalty = beta * sum(m.z[k, t] for k in m.K for t in m.T)

        return production_value - demand_variability_penalty - nonpreferred_line_penalty

    m.Obj = Objective(rule=objective_rule, sense=maximize)

    return m

In [ ]:
# ---- CHECK EACH SKU IS PRODUCIBLE ----
missing = []
for p in P:
    best = max(float(yld.get((str(k), str(p)), 0.0)) for k in K)
    if best <= 0:
        missing.append(p)

print("SKUs with no producing pattern:", missing)

SKUs with no producing pattern: []


In [ ]:
model = build_model(
    P, T, K, L,
    WIP=wip,
    D=demand,
    Y=Y,
    V=value_per_lb,
    R=rate,
    H=hours,
    A=A,
    L_delay=L_delay,
    gamma=gamma,
    beta=beta
)

solver = SolverFactory("highs")
res = solver.solve(model)

m = model

Exports Results to CSV

In [ ]:
import pandas as pd
from pathlib import Path
from pyomo.environ import value
from IPython.display import display

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Guard: make sure solution is loaded ---
print("Termination:", getattr(res, "termination_condition", "UNKNOWN"))

sample_l = next(iter(m.L))
sample_k = next(iter(m.K))
sample_t = next(iter(m.T))

sample_val = m.x[sample_k, sample_l, sample_t].value
if sample_val is None:
    raise ValueError(
        "Model variables have no values (solution not loaded). "
        "Re-solve the model and make sure the solution is loaded."
    )


# 1) NONZERO X VALUES
x_rows = []
for t in sorted(m.T):
    for l in sorted(m.L):
        for k in sorted(m.K):
            v = float(value(m.x[k, l, t]))
            if v > 1e-6:
                x_rows.append({
                    "shift": int(t),
                    "line": str(l),
                    "pattern": str(k),
                    "input_lbs": round(v, 3),
                })

x_df = pd.DataFrame(x_rows)

if x_df.empty:
    x_df = pd.DataFrame(columns=["shift", "line", "pattern", "input_lbs"])
else:
    x_df = x_df.sort_values(["shift", "line", "pattern"])

x_df.to_csv(OUTPUT_DIR / "x_long_nonzero.csv", index=False)
print("Wrote:", (OUTPUT_DIR / "x_long_nonzero.csv").resolve())


# 2) LINE SCHEDULE
if x_df.empty:
    line_schedule = pd.DataFrame(columns=["shift", "line", "cuts"])
else:
    line_sched_rows = []
    for (shift, line), g in x_df.groupby(["shift", "line"]):
        cuts = ", ".join(
            f"{r.pattern} ({r.input_lbs:.0f} lbs)"
            for r in g.sort_values("input_lbs", ascending=False).itertuples(index=False)
        )
        line_sched_rows.append({
            "shift": int(shift),
            "line": str(line),
            "cuts": cuts
        })
    line_schedule = pd.DataFrame(line_sched_rows).sort_values(["shift", "line"])

line_schedule.to_csv(OUTPUT_DIR / "line_schedule.csv", index=False)
print("Wrote:", (OUTPUT_DIR / "line_schedule.csv").resolve())
display(line_schedule.head(20))


# 3) PATTERN MIX BY SHIFT
if x_df.empty:
    pat_mix_pivot = pd.DataFrame(columns=["shift"])
else:
    pat_shift = x_df.groupby(["shift", "pattern"], as_index=False)["input_lbs"].sum()
    pat_pivot = pat_shift.pivot(index="shift", columns="pattern", values="input_lbs").fillna(0.0)
    pat_pivot["TOTAL_LBS"] = pat_pivot.sum(axis=1)

    pct = pat_pivot.drop(columns=["TOTAL_LBS"]).div(
        pat_pivot["TOTAL_LBS"].replace(0, 1), axis=0
    ) * 100
    pct.columns = [f"{c}_pct" for c in pct.columns]

    pat_mix_pivot = pd.concat([pat_pivot.round(2), pct.round(1)], axis=1).reset_index()

pat_mix_pivot.to_csv(OUTPUT_DIR / "pattern_mix_by_shift.csv", index=False)
print("Wrote:", (OUTPUT_DIR / "pattern_mix_by_shift.csv").resolve())
display(pat_mix_pivot.head(10))

# 4) LINE LOAD BY SHIFT
line_load_rows = []
for t in sorted(m.T):
    for l in sorted(m.L):
        sub = x_df[(x_df["shift"] == int(t)) & (x_df["line"] == str(l))] if not x_df.empty else pd.DataFrame()

        total_lbs = float(sub["input_lbs"].sum()) if not sub.empty else 0.0

        hours_used = 0.0
        if not sub.empty:
            for k in sub["pattern"].unique():
                lbs_k = float(sub.loc[sub["pattern"] == k, "input_lbs"].sum())
                hours_used += lbs_k / float(rate[str(k)])

        hrs_avail = float(hours[(str(l), int(t))])
        util = (hours_used / hrs_avail * 100.0) if hrs_avail > 0 else 0.0

        line_load_rows.append({
            "shift": int(t),
            "line": str(l),
            "total_input_lbs": round(total_lbs, 2),
            "hours_used": round(hours_used, 3),
            "hours_available": round(hrs_avail, 3),
            "util_pct": round(util, 1),
        })

line_load = pd.DataFrame(line_load_rows).sort_values(["shift", "line"])
line_load.to_csv(OUTPUT_DIR / "line_load_by_shift.csv", index=False)
print("Wrote:", (OUTPUT_DIR / "line_load_by_shift.csv").resolve())
display(line_load.head(20))


# 5) PRODUCTION VS DEMAND BY SHIFT
prod_rows = []
for t in sorted(m.T):
    for p in sorted(P):
        produced = float(value(m.prod[p, t]))
        dem = float(demand.get((str(p), int(t)), 0.0))

        prod_rows.append({
            "shift": int(t),
            "sku": str(p),
            "produced_lbs": round(produced, 3),
            "demand_lbs": round(dem, 3),
            "over_under": round(produced - dem, 3),
        })

prod_df = pd.DataFrame(prod_rows).sort_values(["shift", "sku"])
prod_df.to_csv(OUTPUT_DIR / "production_vs_demand_by_shift.csv", index=False)
print("Wrote:", (OUTPUT_DIR / "production_vs_demand_by_shift.csv").resolve())
display(prod_df.head(25))

print("\nAll outputs written to:", OUTPUT_DIR.resolve())

Termination: UNKNOWN
Wrote: /content/outputs/x_long_nonzero.csv
Wrote: /content/outputs/line_schedule.csv


,shift,line,cuts
0,1,Line1,P1 (48000 lbs)
1,1,Line2,"P1 (41311 lbs), P11 (2551 lbs), P10 (1374 lbs)..."
2,1,Line3,"P1 (44409 lbs), P3 (1701 lbs), P10 (1374 lbs)"
3,2,Line1,"P1 (42873 lbs), P10 (2747 lbs), P4 (1391 lbs)"
4,2,Line2,"P1 (47233 lbs), P11 (562 lbs)"
5,2,Line3,"P1 (47233 lbs), P12 (549 lbs)"
6,3,Line1,"P1 (43913 lbs), P10 (2747 lbs), P4 (403 lbs)"
7,3,Line2,"P1 (45378 lbs), P8 (1766 lbs), P2 (452 lbs)"
8,3,Line3,"P1 (45378 lbs), P3 (1701 lbs), P11 (633 lbs)"
9,4,Line1,P1 (48000 lbs)


Wrote: /content/outputs/pattern_mix_by_shift.csv


,shift,P1,P10,P11,P12,P2,P3,P4,P8,TOTAL_LBS,P1_pct,P10_pct,P11_pct,P12_pct,P2_pct,P3_pct,P4_pct,P8_pct
0,1,133720.12,2747.25,2551.02,0.00,452.49,1700.68,0.00,750.30,141921.86,94.2,1.9,1.8,0.0,0.3,1.2,0.0,0.5
1,2,137339.17,2747.25,562.23,549.45,0.00,0.00,1391.25,0.00,142589.35,96.3,1.9,0.4,0.4,0.0,0.0,1.0,0.0
2,3,134667.83,2747.25,632.88,0.00,452.49,1700.68,403.26,1765.79,142370.17,94.6,1.9,0.4,0.0,0.3,1.2,0.3,1.2
3,4,133720.12,2747.25,2551.02,0.00,452.49,1700.68,0.00,750.30,141921.86,94.2,1.9,1.8,0.0,0.3,1.2,0.0,0.5
4,5,133539.69,2747.25,2551.02,549.45,452.49,1478.28,0.00,459.42,141777.60,94.2,1.9,1.8,0.4,0.3,1.0,0.0,0.3
5,6,134005.93,2747.25,2551.02,549.45,0.00,1478.28,528.64,0.00,141860.58,94.5,1.9,1.8,0.4,0.0,1.0,0.4,0.0
6,7,134180.27,2747.25,2551.02,0.00,0.00,1700.68,0.00,750.30,141929.52,94.5,1.9,1.8,0.0,0.0,1.2,0.0,0.5
7,8,136818.35,2747.25,1290.17,0.00,0.00,1700.68,0.00,0.00,142556.45,96.0,1.9,0.9,0.0,0.0,1.2,0.0,0.0
8,9,133245.41,2747.25,2551.02,549.45,452.49,1478.28,279.56,459.42,141762.89,94.0,1.9,1.8,0.4,0.3,1.0,0.2,0.3
9,10,141467.59,0.00,0.00,907.44,1245.10,0.00,0.00,0.00,143620.14,98.5,0.0,0.0,0.6,0.9,0.0,0.0,0.0


Wrote: /content/outputs/line_load_by_shift.csv


,shift,line,total_input_lbs,hours_used,hours_available,util_pct
0,1,Line1,48000.00,16.0,16.0,100.0
1,1,Line2,46438.38,16.0,16.0,100.0
2,1,Line3,47483.48,16.0,16.0,100.0
3,2,Line1,47011.03,16.0,16.0,100.0
4,2,Line2,47795.55,16.0,16.0,100.0
5,2,Line3,47782.78,16.0,16.0,100.0
6,3,Line1,47063.03,16.0,16.0,100.0
7,3,Line2,47595.93,16.0,16.0,100.0
8,3,Line3,47711.22,16.0,16.0,100.0
9,4,Line1,48000.00,16.0,16.0,100.0


Wrote: /content/outputs/production_vs_demand_by_shift.csv


,shift,sku,produced_lbs,demand_lbs,over_under
0,1,1001,114999.299,1714.286,113285.013
1,1,1023,1428.571,1428.571,0.000
2,1,2021,1428.571,1428.571,0.000
3,1,3033,1256.002,1357.143,-101.140
4,1,4043,714.286,714.286,0.000
5,1,5055,714.286,714.286,0.000
6,1,6066,549.451,714.286,-164.835
7,1,7072,714.286,714.286,0.000
8,2,1001,118111.685,1714.286,116397.399
9,2,1023,1043.956,1428.571,-384.615



All outputs written to: /content/outputs
